# GARCH Model Tutorial

This notebook demonstrates how to model **time-varying volatility** with a
**Generalized Autoregressive Conditional Heteroskedasticity (GARCH)** model.

## What is GARCH?

A GARCH(p, q) model captures **volatility clustering** — periods of high
volatility tend to follow other high-volatility periods.

$$\sigma_t^2 = \omega + \sum_{i=1}^{q} \alpha_i \varepsilon_{t-i}^2 + \sum_{j=1}^{p} \beta_j \sigma_{t-j}^2$$

where $\varepsilon_t = \sigma_t z_t$ and $z_t \sim N(0,1)$.

## Dataset

We use the **S&P 500 daily returns** sample dataset from the `arch` package.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from timeseries import moving_average, autocorrelation, exponential_smoothing

## 1. Load the data

In [ ]:
from arch.data import sp500

data = sp500.load()
returns = data["Adj Close"].pct_change().dropna() * 100  # percentage returns
returns.name = "S&P 500 Daily Returns (%)"

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(returns)
axes[0].set_title("S&P 500 Daily Returns (%)")
axes[1].plot(returns ** 2)
axes[1].set_title("Squared Returns (proxy for volatility)")
plt.tight_layout()
plt.show()

## 2. Explore with library helpers

In [ ]:
ret_list = returns.values.tolist()

# 21-day (≈1 month) moving average of absolute returns as a volatility proxy
abs_ret = [abs(r) for r in ret_list]
ma21 = moving_average(abs_ret, 21)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(returns.index, abs_ret, alpha=0.3, label="|return|")
ax.plot(returns.index, ma21, label="21-day MA of |return|", linewidth=2)
ax.set_title("Volatility Proxy — 21-Day Moving Average of Absolute Returns")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Exponential smoothing of squared returns
sq_ret = [r ** 2 for r in ret_list]
ema_vol = exponential_smoothing(sq_ret, alpha=0.06)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(returns.index, sq_ret, alpha=0.2, label="Squared returns")
ax.plot(returns.index, ema_vol, label="EMA(0.06) of squared returns", linewidth=2)
ax.set_title("Exponential Smoothing of Squared Returns (timeseries.exponential_smoothing)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation of squared returns — evidence of volatility clustering
lags = range(1, 31)
acf_sq = [autocorrelation(sq_ret, lag) for lag in lags]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lags, acf_sq)
ax.set_xlabel("Lag")
ax.set_ylabel("ACF")
ax.set_title("ACF of Squared Returns — Evidence of Volatility Clustering")
plt.tight_layout()
plt.show()

## 3. Fit a GARCH(1,1) model

In [ ]:
from arch import arch_model

model = arch_model(returns, vol="Garch", p=1, q=1, mean="Constant", dist="Normal")
result = model.fit(disp="off")
print(result.summary())

## 4. Conditional volatility

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(returns, alpha=0.4, label="Returns")
ax.plot(result.conditional_volatility, color="red", linewidth=1.5,
        label="GARCH(1,1) Conditional Vol")
ax.plot(-result.conditional_volatility, color="red", linewidth=1.5)
ax.set_title("S&P 500 Returns with GARCH(1,1) Conditional Volatility Bands")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Volatility forecasting

In [ ]:
forecast = result.forecast(horizon=10)
variance_fc = forecast.variance.iloc[-1]
vol_fc = np.sqrt(variance_fc)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 11), vol_fc.values, marker="o")
ax.set_xlabel("Days Ahead")
ax.set_ylabel("Forecasted Volatility (%)")
ax.set_title("GARCH(1,1) — 10-Day Volatility Forecast")
plt.tight_layout()
plt.show()

## 6. Standardised residual diagnostics

In [ ]:
std_resid = result.resid / result.conditional_volatility

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(std_resid.dropna(), bins=50, density=True, alpha=0.7)
axes[0].set_title("Histogram of Standardised Residuals")

acf_resid_sq = [autocorrelation((std_resid.dropna() ** 2).tolist(), lag) for lag in range(1, 21)]
axes[1].bar(range(1, 21), acf_resid_sq)
axes[1].set_title("ACF of Squared Standardised Residuals")
axes[1].set_xlabel("Lag")
plt.tight_layout()
plt.show()

## Summary

| Step | Tool / Function |
|---|---|
| Volatility proxy | `timeseries.moving_average`, `timeseries.exponential_smoothing` |
| Volatility-clustering evidence | `timeseries.autocorrelation` on squared returns |
| GARCH fitting | `arch.arch_model` |
| Residual diagnostics | `timeseries.autocorrelation` on standardised residuals |